# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup (openWakeWord training - PyTorch path only, Google Colab Python 3.12)

# 1) Piper sample generator - PIN to v2.0.0.
#    train.py does `from generate_samples import generate_samples` with a specific signature that
#    ONLY exists in the v2.0.0 tag. HEAD (v3.2.0+) was restructured into a package with a different
#    API and would break that import. The model wget below already targets v2.0.0.
!git clone --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# PyTorch >=2.6 made torch.load default to weights_only=True, which REJECTS the Piper VITS
# checkpoint (custom class piper_train.vits.models.SynthesizerTrn). Without this, generate_clips
# loads 0 positive clips and steps 2/3 then crash with `ValueError: high <= 0`. Force
# weights_only=False (checkpoint is from the trusted rhasspy release; piper_train ships in this
# repo so it unpickles fine).
!sed -i 's/model = torch.load(model_path)/model = torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py

# 2) TTS phonemizer + VAD for clip generation.
#    piper-phonemize 1.1.0 has NO cp312 Linux wheel and no sdist -> `pip install piper-phonemize`
#    FAILS on Colab Py3.12. piper-phonemize-fix installs the SAME top-level module `piper_phonemize`
#    (exports phonemize_espeak) and ships a cp312 manylinux wheel.
#    webrtcvad 2.0.10 is sdist-only (compiles a C ext); webrtcvad-wheels has prebuilt cp312 wheels
#    with the identical `import webrtcvad` API.
!pip install piper-phonemize-fix==1.2.2
!pip install webrtcvad-wheels==2.0.14

# 3) openWakeWord (editable) - provides openwakeword.data / openwakeword.utils used by train.py.
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# 4) Audio + training deps actually imported by train.py / data.py.
#    torch-audiomentations BUMPED 0.11.0 -> 0.11.1: 0.11.0 calls torchaudio.set_audio_backend(),
#    REMOVED in torchaudio>=2.1, which crashed `import torch_audiomentations` at module load
#    (the blocker for steps 1/2/3). 0.11.1 deleted that line upstream and is API-compatible.
#    A normal (NOT --no-deps) install also pulls julius/librosa/torch-pitch-shift, which the
#    package imports at load time. torch-audiomentations has no upper bound on torch, so Colab's
#    torch 2.x is kept (no downgrade / runtime restart).
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.1
!pip install julius librosa   # safety net: torch_audiomentations imports these at load time
!pip install acoustics==0.2.6
!pip install pronouncing==0.2.0
!pip install deep-phonemizer==0.0.19   # OOV fallback in generate_adversarial_texts (`from dp.phonemizer import Phonemizer`)
!pip install onnxscript   # recent torch.onnx.export routes through the new exporter, which imports onnxscript (pulls onnx); needed to export the trained .onnx
!pip install "datasets>=2.16,<3"        # relaxed from the stale 2.14.6 pin so it resolves on current Colab

# DROPPED (TFLite-only, gated behind --convert_to_tflite which this notebook never sets, AND
#          uninstallable on Python 3.12 -> they were the original cell's install failure):
#   tensorflow-cpu==2.8.1, tensorflow_probability==0.16.0, onnx_tf==1.10.0
# The training path emits <model_name>.onnx via torch.onnx.export (bundled with torch); no TF needed.

# Download required openWakeWord feature models (melspectrogram + embedding) used by AudioFeatures().
# They ship EMPTY in the repo; without them the augment/train steps cannot load the onnxruntime models.
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# DeepPhonemizer (dp) ALSO calls torch.load() without weights_only -> same PyTorch >=2.6
# failure when generate_clips builds negative adversarial texts for OOV words (e.g. "maximo").
# Patch it the same way (its checkpoint is the trusted DeepPhonemizer release).
import dp
_dp_model = os.path.join(os.path.dirname(dp.__file__), "model", "model.py")
_dp_src = open(_dp_model).read().replace(
    "torch.load(checkpoint_path, map_location=device)",
    "torch.load(checkpoint_path, map_location=device, weights_only=False)")
open(_dp_model, "w").write(_dp_src)

# torchaudio on recent Colab removed the top-level `torchaudio.info`; torch_audiomentations
# (AddBackgroundNoise) and openwakeword data.py call it during augment_clips -> AttributeError.
# torchaudio.load still works, so shim ONLY info, backed by soundfile (installed via librosa).
# Inject the shim into train.py (which runs as a subprocess) right after its imports so it covers
# every caller in that process.
_tp = "openwakeword/openwakeword/train.py"
_tsrc = open(_tp).read()
_shim = (
    "\nimport torchaudio as _ta\n"
    "if not hasattr(_ta, 'info'):\n"
    "    import soundfile as _sf\n"
    "    class _AInfo:\n"
    "        def __init__(self, num_frames, sample_rate, num_channels):\n"
    "            self.num_frames = num_frames; self.sample_rate = sample_rate\n"
    "            self.num_channels = num_channels; self.bits_per_sample = 16\n"
    "    def _ta_info(uri, *a, **k):\n"
    "        f = _sf.info(str(uri)); return _AInfo(f.frames, f.samplerate, f.channels)\n"
    "    _ta.info = _ta_info\n"
)
_anchor = "from openwakeword.utils import AudioFeatures\n"
if "_ta_info" not in _tsrc and _anchor in _tsrc:
    open(_tp, "w").write(_tsrc.replace(_anchor, _anchor + _shim, 1))

# Skip the optional ONNX->TFLite conversion. train.py declares --convert_to_tflite with a buggy
# default="False" (a truthy STRING) and checks `if args.convert_to_tflite:`, so the tflite step
# runs even when NOT requested and crashes on the (intentionally dropped) onnx_tf/tensorflow deps
# AFTER the .onnx has already been written. Require the flag explicitly so --train_model ends clean.
_tp2 = "openwakeword/openwakeword/train.py"
_s2 = open(_tp2).read()
open(_tp2, "w").write(_s2.replace("if args.convert_to_tflite:", "if args.convert_to_tflite is True:"))

# Make the exported ONNX self-contained. The current torch ONNX exporter writes the weights to
# an external <model>.onnx.data file (the .onnx is then just the graph, a few KB). That's awkward
# to ship to a single-file consumer (e.g. flutter_onnxruntime asset). Re-save the model embedded
# right after export so <model>.onnx is one self-contained file. onnx is available via onnxscript.
_tp4 = "openwakeword/openwakeword/train.py"
_s4 = open(_tp4).read()
_anchor4 = 'os.path.join(output_dir, model_name + ".onnx"), opset_version=13)\n'
_embed4 = (_anchor4 +
    '        import onnx as _onnx_embed\n'
    '        _embed_path = os.path.join(output_dir, model_name + ".onnx")\n'
    '        _onnx_embed.save_model(_onnx_embed.load(_embed_path), _embed_path, save_as_external_data=False)\n')
if "save_as_external_data=False" not in _s4 and _anchor4 in _s4:
    open(_tp4, "w").write(_s4.replace(_anchor4, _embed4, 1))


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download background audio (ESC-50 environmental sounds)
#
# NOTE: this replaces the FMA `datasets` streaming download, which on current Colab raises
# "ValueError: Cannot seek streaming HTTP file" (a datasets/fsspec streaming bug with the
# rudraml/fma dataset). ESC-50 is a small (~600 MB) set of 2000 environmental clips, fetched as a
# plain zip (NO streaming) and resampled to 16k mono. Output stays in ./fma so the config's
# background_paths=['./fma'] keeps working. (AudioSet was already dropped: its HF tar URLs 404
# after the parquet migration.)

import os, glob, librosa, numpy as np, scipy.io.wavfile

output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)

!wget -q -O esc50.zip https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip
!unzip -q -o esc50.zip

clips = glob.glob("ESC-50-master/audio/*.wav")
print(f"resampling {len(clips)} ESC-50 clips to 16k mono ...")
for f in tqdm(clips):
    y, sr = librosa.load(f, sr=16000, mono=True)  # decode + resample to 16k; no HTTP seek
    name = os.path.basename(f)
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (y * 32767).astype(np.int16))


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

# --- Wake word + pronunciation variants --------------------------------------
# openWakeWord is an ACOUSTIC matcher on an English-trained embedding: it has no
# concept of language. To cover different accents / pronunciations ("tonalita'")
# we feed SEVERAL phonetic spellings of the SAME phrase. train.py passes the whole
# target_phrase list to Piper TTS (which also varies speed via the hardcoded
# length_scales=[0.75, 1.0, 1.25] and uses the ~900-speaker libritts_r voices), so
# the positive set spans the ways the word is actually said. Edit these two lines
# to retrain another word, just change WAKE_WORD and PRON_VARIANTS below.
WAKE_WORD = "nexo"
PRON_VARIANTS = [
    "nexo", "necso", "nekso",      # EN / hard "ks" renderings
    "nehkso", "nexxo", "nexso",    # open "e", stressed, IT "x"
]

config["target_phrase"] = PRON_VARIANTS
config["model_name"] = WAKE_WORD.replace(" ", "_")

# --- Dataset size: the single biggest lever on recall + accent robustness -----
# The original 1,000 was a "save time for the example" value and gave recall ~0.5.
# openWakeWord recommends >= 20,000; 10,000 is the principled default and ~10x the
# data while staying feasible on a free Colab T4 (generation ~1.5-2 h). Bump higher
# (20k-30k) if you have Colab Pro / more time.
# NOTE: a single short word like "nexo" (no "hey" prefix) is acoustically easier
# to false-trigger; keep the FP/hour target tight and test the threshold in-app.
config["n_samples"] = 10000
config["n_samples_val"] = 2000
config["augmentation_rounds"] = 2          # more room/noise variety per clip
config["steps"] = 20000

# --- Stop criteria: don't settle for a weak model ----------------------------
# The 0.25 recall target made the trainer stop optimizing recall far too early.
config["target_accuracy"] = 0.7
config["target_recall"] = 0.6
config["target_false_positives_per_hour"] = 0.2

# --- Confusables to suppress (fewer false activations) -----------------------
config["custom_negative_phrases"] = [
    "next", "nexus", "echo", "exo", "neck", "necks",
]

# --- Augmentation / negative data (unchanged paths) --------------------------
config["background_paths"] = ['./fma']  # ESC-50 environmental sounds (downloaded above)
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Step 4 (OPTIONAL - NOT NEEDED): convert the .onnx to .tflite.
# SKIP THIS CELL. The training step already produced my_custom_model/<model_name>.onnx via
# torch.onnx.export, which is what we use (flutter_onnxruntime / Home Assistant consume ONNX).
# This conversion needs onnx + tensorflow + onnx_tf, which we intentionally DROPPED from the env
# setup because they don't install on Python 3.12. Running it WILL fail with
# `ModuleNotFoundError: No module named 'onnx'` - that's expected; just leave it unrun.
# (If you ever truly need .tflite, convert in a separate Python 3.10 env or use onnx2tf.)

# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!